### Assignment 1 
- GACS 7203
- Domenica Burroughs
- October 10, 2026

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
###########################
# Listing 1: Loading and imputing the data
########################### 

# load the csv file
data = pd.read_csv(’datasetPISA.csv’)
cols = data.columns

# imputation
imputer = KNNImputer(n_neighbors=11)
colImp = [cols[1]] + list(cols[4:])
data[colImp] = imputer.fit_transform(data[colImp])

# using GDP per capita and corruption index for the exercise
colAnalysis = [cols[4], cols[5]]

# scale features to [0, 1]
scaler = MinMaxScaler()
data.loc[:, colAnalysis] = scaler.fit_transform(data[colAnalysis])

In [ ]:
###########################
# Listing 2: Evaluate Quatratic Fcn. 
###########################
# g_i(x) = x^T W_i x + w_i^T x + w_i0 (2D case)
# Boundary: g_i(x) = g_j(x). For fixed x1, solve quadratic in x2.

def funQuadratic_evalPoint(Wi, wi, wi0, Wj, wj, wj0, x1):
    # shape checks (fail fast)
    assert Wi.shape == (2, 2) and Wj.shape == (2, 2)
    wi = np.asarray(wi).reshape(2,)
    wj = np.asarray(wj).reshape(2,)

    dW = Wi - Wj            # 2x2
    dw = wi - wj            # (2,)
    dw0 = float(wi0 - wj0)  #scalar

    # a*x2^2 + b*x2 + c = 0
    a = dW[1, 1]
    b = 2.0 * dW[0, 1] * x1 + dw[1]
    c = dW[0, 0] * (x1**2) + dw[0] * x1 + dw0

    # handle linear/degenerate case
    if abs(a) < 1e-12:
        if abs(b) < 1e-12:
            return np.nan, np.nan
        y = -c / b
        return y, y

    disc = b * b - 4.0 * a * c
    if disc < 0:
        return np.nan, np.nan

    r = np.sqrt(disc)
    y1 = (-b + r) / (2.0 * a)
    y2 = (-b - r) / (2.0 * a)
    return y1, y2

def funQuadratic_eval(Wi, wi, wi0, Wj, wj, wj0, n_points=200):
    x = np.linspace(0.0, 1.0, n_points)
    y1 = np.empty_like(x)
    y2 = np.empty_like(x)
    for i, x1 in enumerate(x):
        y1[i], y2[i] = funQuadratic_evalPoint(Wi, wi, wi0, Wj, wj, wj0, x1)
    return x, y1, y2

In [ ]:
###########################
# A) Compute the mean and covariance using maximum likelihood
###########################
# Extract features
X = data[colAnalysis].values

# Replace "Class" with the actual class column name
y = data["Class"].values

classes = np.unique(y)

means = {}
covariances = {}

for c in classes:
    Xc = X[y == c]
    means[c] = np.mean(Xc, axis=0)
    # Maximum likelihood covariance estimate
    covariances[c] = np.cov(Xc, rowvar=False, bias=True)

    print(f"\nClass {c}")
    print("Mean:")
    print(means[c])

    print("\nCovariance:")
    print(covariances[c])

In [ ]:
###########################
# B) Create Wi, wi and wi0
###########################

In [ ]:
###########################
# C) Predict Every Sample
###########################
# Create descriminant function
def discriminant(x, Wi, wi, wi0):
    return (
        x.T @ Wi @ x
        + wi.T @ x
        + wi0
    )

# Predict classes
predicted = []

for x in X:
    scores = []
    for c in classes:
        g = discriminant(x, Wi[c], wi[c], wi0[c])
        scores.append(g)
    predicted.append(classes[np.argmax(scores)])
predicted = np.array(predicted)

In [ ]:
###########################
# D) Scatter Plot and Decision Boundaries
###########################
markers = {
    0: 'o',
    1: 'x',
    2: '^'
}

for c in classes:

    Xc = X[y == c]

    plt.scatter(
        Xc[:,0],
        Xc[:,1],
        marker=markers[c],
        label=f'Class {c}'
    )

for i, j in pairs:
    xb, y1, y2 = funQuadratic_eval(
        Wi[i], wi[i], wi0[i],
        Wi[j], wi[j], wi0[j]
    )

    plt.plot(xb, y1, 'k')
    plt.plot(xb, y2, 'k')

# Figure Labels
plt.xlabel("GDP per capita")
plt.ylabel("Corruption Index")
plt.title("PISA Classification")

plt.legend()
plt.grid(True)
plt.show()